# 第 6 章 · GRPO 为什么不适合底层运控：整条打分 vs 逐步打分

配套讲义：仓库里 `robot-motion-rl/notes/lessons/06-grpo-credit.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（轨迹是编的，critic 假设理想）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 GRPO | (r − mean)/std | `grpo_advantages([3, 6, 6, 6])` | [−1.732, 0.577, 0.577, 0.577] |
| 第三步 理想 V | 还剩几步 | `ideal_values(3, 6, mistake=1)` | [6, 5, 1, 0] |
| 第三步 δ | r + V' − V | `td_errors([1, 1, 1], v)` | [0, −3, 0] |

例子（编的）：4 条轨迹、最长 6 步、每步站着得 1 分；轨迹 A 在第 1 步踩错，第 3 步后摔倒（只活 3 步）。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `grpo_advantages`**：GRPO：A_i = (R_i − 组平均) / 组标准差，整条轨迹每一步都用这一个数。

In [ ]:
def grpo_advantages(returns):
    """GRPO：A_i = (R_i − 组平均) / 组标准差，整条轨迹每一步都用这一个数。"""
    r = np.asarray(returns, float)
    return (r - r.mean()) / r.std()

**函数 `td_errors`**：GAE 取 λ=0、γ=1：A_t = δ_t = r_t + V(s_{t+1}) − V(s_t)。values 比 rewards 多一个（最后是 0）。

In [ ]:
def td_errors(rewards, values):
    """GAE 取 λ=0、γ=1：A_t = δ_t = r_t + V(s_{t+1}) − V(s_t)。values 比 rewards 多一个（最后是 0）。"""
    return [rewards[t] + values[t + 1] - values[t] for t in range(len(rewards))]

**函数 `ideal_values`**：理想 critic：没出错时 V = 还剩几步；踩错之后 V = 摔倒前还剩几步。

In [ ]:
def ideal_values(length, horizon, mistake=None):
    """理想 critic：没出错时 V = 还剩几步；踩错之后 V = 摔倒前还剩几步。"""
    v = []
    for t in range(length + 1):
        if t >= length:
            v.append(0.0)
        elif mistake is not None and t > mistake:
            v.append(float(length - t))
        else:
            v.append(float(horizon - t))
    return v

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 GRPO 整条打分")
returns = [3, 6, 6, 6]
adv = grpo_advantages(returns)
print("回报 =", returns, "→ 优势 =", np.round(adv, 3))

In [ ]:
print("\n第二步 PPO + GAE 逐步打分（轨迹 A）")
v = ideal_values(3, 6, mistake=1)
print("V =", v)
print("A_t =", td_errors([1, 1, 1], v))

In [ ]:
print("\n第三步 惩罚落在踩错那一步的比例")
print(f"GRPO：1/3 = {1 / 3:.0%}；PPO + GAE：3/3 = 100%")

## 3. 代码里才看得到的

- `np.std` 默认除以 G（总体标准差），和第 24 章的算法一致；有些实现除以 G − 1，数会稍大。
- 理想 critic 在踩错的**下一个**状态才变成「注定要摔」，所以 δ 的负值出现在踩错那一步（它连接好状态和坏状态）。

## 4. 自己改一改

- 把轨迹 A 改成第 4 步才踩错，看 GRPO 和 δ 怎么变。
- 给理想 V 加一点噪声（±0.5），看 δ 的红色会不会「散开」。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 同题答几遍，组平均当 baseline，整条共享一个分；
> 踩空一脚晃几下才摔，错在那一脚，不在后面晃；
> critic 看出局面何时变坏，惩罚落在那一步；
> 长时域稠密奖励用 PPO，成功失败一个分才轮到 GRPO。